# 🚗 AI-Based Smart Vehicle Insurance Claim Assessment System
## Official Deep Learning Model Training Pipeline (Google Colab T4 GPU)

This notebook provides the **genuine, end-to-end training and export pipeline** for the two neural networks powering the automated insurance assessment system:
1. **Vehicle Body Parts Segmentation Model** (`yolov8s-seg` fine-tuned on 3,833 authentic vehicle parts images across 9 classes)
2. **Vehicle Damage Localization & Segmentation Model** (`yolov8s-seg` fine-tuned on real CarDD dataset across 6 damage classes)

⚡ **Hardware Requirement**: Google Colab Free Tier with **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

### Step 1: Verify Hardware & GPU Acceleration

In [ ]:
# Check NVIDIA GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2: Install Dependencies & Clone Repository

In [ ]:
# Install ultralytics (YOLOv8), ONNX, and OpenCV
!pip install -q ultralytics onnx onnxruntime opencv-python pyyaml

import os
import shutil
from pathlib import Path

# Clone project repository
REPO_URL = "https://github.com/Aryan00Saini/AI-Based-Smart-Vehicle-Insurance-Claim-Assessment-System.git"
REPO_DIR = Path("/content/AI-Based-Smart-Vehicle-Insurance-Claim-Assessment-System")

if not REPO_DIR.exists():
    print(f"Cloning project repository from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present. Pulling latest updates...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

### Step 3: Prepare Real Datasets
- Ingests the **real Ultralytics Carparts dataset** (3,833 vehicle photographs) and remaps to our 9-class parts taxonomy.
- Ingests the **real CarDD dataset** (instance segmentations) and remaps to our 6-class damage taxonomy.

In [ ]:
# Generate dataset configuration YAMLs
!python training/dataset_prep.py

# Download carparts dataset if not already extracted
carparts_zip = Path("/content/carparts-seg.zip")
if not Path("datasets/carparts-seg").exists():
    print("Downloading Ultralytics Carparts segmentation dataset...")
    !wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/carparts-seg.zip -O {carparts_zip}
    !unzip -q {carparts_zip} -d datasets/
    print("Carparts dataset extracted successfully!")

# Remap Carparts into 9-class vehicle parts taxonomy
!python training/convert_carparts.py

# Convert CarDD dataset into 6-class damage taxonomy
# On Colab's high-speed connection, we can ingest 500+ real CarDD damage samples
!python training/convert_cardd.py

### Step 4: Fine-Tune YOLOv8s-seg for Vehicle Parts Segmentation (9 Classes)
**Taxonomy:** `bumper_front`, `bumper_rear`, `door`, `fender`, `headlamp`, `taillamp`, `mirror`, `hood`, `windshield`

In [ ]:
from ultralytics import YOLO

print("Starting YOLOv8s-seg Parts Segmentation Training...")
parts_model = YOLO("yolov8s-seg.pt")

parts_results = parts_model.train(
    data="training/parts_dataset.yaml",
    epochs=50,          # Set to 50-100 for production grade convergence
    imgsz=640,
    batch=16,
    device=0,           # GPU 0
    project="training/runs",
    name="parts_segmentation",
    exist_ok=True,
    verbose=True
)

print("Parts training complete!")

### Step 5: Fine-Tune YOLOv8s-seg for Damage Segmentation (6 Classes)
**Taxonomy:** `scratch`, `dent`, `crack`, `shatter`, `paint_chip`, `misalignment`

In [ ]:
print("Starting YOLOv8s-seg Damage Segmentation Training on CarDD...")
damage_model = YOLO("yolov8s-seg.pt")

damage_results = damage_model.train(
    data="training/damage_dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="training/runs",
    name="damage_segmentation",
    exist_ok=True,
    verbose=True
)

print("Damage training complete!")

### Step 6: Validate & Display Evaluation Curves (For Project Report & Viva)
Plots training loss, mAP50, and mAP50-95 curves and confusion matrices.

In [ ]:
from IPython.display import Image, display

print("--- PARTS SEGMENTATION METRICS ---")
parts_metrics_img = Path("training/runs/parts_segmentation/results.png")
if parts_metrics_img.exists():
    display(Image(str(parts_metrics_img), width=800))

parts_cm = Path("training/runs/parts_segmentation/confusion_matrix.png")
if parts_cm.exists():
    display(Image(str(parts_cm), width=700))

print("\n--- DAMAGE SEGMENTATION METRICS ---")
damage_metrics_img = Path("training/runs/damage_segmentation/results.png")
if damage_metrics_img.exists():
    display(Image(str(damage_metrics_img), width=800))

damage_cm = Path("training/runs/damage_segmentation/confusion_matrix.png")
if damage_cm.exists():
    display(Image(str(damage_cm), width=700))

### Step 7: Export Best Checkpoints to Production ONNX (Opset 17)

In [ ]:
# Run the production export script
!python training/export_models.py

# Package models and training evaluation graphs into a zip file for download
!zip -j /content/trained_models.zip \
    data/models/yolov8s_parts.onnx \
    data/models/yolov8s_damage.onnx \
    training/runs/parts_segmentation/results.png \
    training/runs/damage_segmentation/results.png

print("\n✅ Trained models and evaluation graphs packaged in /content/trained_models.zip")

# Trigger Colab browser download
from google.colab import files
files.download("/content/trained_models.zip")